# MSDM5058 Financial Time Series Analysis

**Project I: Time Series Analysis with Financial Data**

This notebook performs comprehensive time series analysis on two stock price series.

- **Stocks**: AAPL (Apple Inc.) and MSFT (Microsoft Corp.)
- **Data Source**: Yahoo Finance
- **Analysis Period**: 15+ years (≥4000 trading days)

## Setup and Imports

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Import custom modules
import sys
sys.path.append('..')

from src.preprocessing import download_data, load_data, compute_returns, demean, preprocess_data, plot_prices_and_returns
from src.stationarity import adf_test, plot_acf_pacf, plot_acf_absolute, compare_linear_nonlinear_acf, select_arma_order, fit_arma, compare_arma
from src.fractal import hurst_exponent, plot_hurst, dfa_analysis, plot_dfa, multifractal_analysis, plot_multifractal, mfdfa_analysis, plot_mfdfa
from src.granger import prepare_bivariate_series, select_varma_order, varma_model, get_coefficients, granger_causality_test, plot_granger_results
from src.fourier import fourier_transform, power_spectrum, plot_fourier, plot_power_spectrum, analyze_spectral_properties
from src.emd import emd_decomposition, plot_imfs, plot_selected_imfs, compute_imf_hurst, plot_hurst_imf, analyze_imf_psd, analyze_reduced_series, analyze_imfs

---
# Part 1: Data Preprocessing

Download stock data and compute daily log returns.

In [ ]:
# Configuration
TICKERS = ['AAPL', 'MSFT']
START_DATE = '2009-01-01'
END_DATE = '2024-04-01'
DATA_DIR = '../data'

print(f"Analyzing stocks: {TICKERS}")
print(f"Period: {START_DATE} to {END_DATE}")

In [ ]:
# Download data from Yahoo Finance
# Uncomment the following line to download fresh data
# data = download_data(TICKERS, START_DATE, END_DATE, DATA_DIR)

# Or load existing data
data = load_data(TICKERS, DATA_DIR)

In [ ]:
# Compute returns and preprocess
prices = {}
returns = {}
X = {}  # Demeaned returns

for ticker in TICKERS:
    prices[ticker], returns[ticker], X[ticker] = preprocess_data(data[ticker])

In [ ]:
# Plot prices and returns
for ticker in TICKERS:
    fig = plot_prices_and_returns(prices[ticker], X[ticker], ticker)
    plt.savefig(f'../figures/{ticker}_prices_returns.png', dpi=150, bbox_inches='tight')
    plt.show()

---
# Part 2: Stationarity and Autocorrelation

Perform ADF test and analyze autocorrelation functions.

## 2.1 Augmented Dickey-Fuller Test

In [ ]:
# ADF test for stationarity
adf_results = {}
for ticker in TICKERS:
    adf_results[ticker] = adf_test(X[ticker], name=ticker)

## 2.2 ACF and PACF Analysis

In [ ]:
# Plot ACF and PACF
for ticker in TICKERS:
    fig = plot_acf_pacf(X[ticker], lags=50, title=f'{ticker} ')
    plt.savefig(f'../figures/{ticker}_acf_pacf.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2.3 ARMA Model Fitting

In [ ]:
# Select optimal ARMA order and compare with guess
# Based on ACF/PACF, make an initial guess
guess_orders = {'AAPL': (1, 1), 'MSFT': (1, 1)}  # Modify based on your ACF/PACF analysis

arma_results = {}
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"ARMA Analysis for {ticker}")
    print(f"{'='*60}")
    
    arma_results[ticker] = compare_arma(X[ticker], guess_orders[ticker])

## 2.4 Linear vs Nonlinear Autocorrelation

In [ ]:
# Plot ACF of |X| (nonlinear autocorrelation)
for ticker in TICKERS:
    fig = plot_acf_absolute(X[ticker], lags=50, title=f'{ticker} ')
    plt.savefig(f'../figures/{ticker}_nonlinear_acf.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Compare linear and nonlinear
    comparison = compare_linear_nonlinear_acf(X[ticker], lags=50)

---
# Part 3: Fractal Behavior Analysis

Analyze fractality using Hurst exponent, DFA, and multifractal methods.

## 3.1 Hurst Exponent

In [ ]:
# Compute Hurst exponent
hurst_results = {}
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Hurst Exponent Analysis for {ticker}")
    print(f"{'='*60}")
    
    hurst_results[ticker] = hurst_exponent(X[ticker])
    
    # Plot R/S analysis
    fig = plot_hurst(hurst_results[ticker])
    plt.savefig(f'../figures/{ticker}_hurst.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3.2 Detrended Fluctuation Analysis (DFA)

In [ ]:
# DFA analysis
dfa_results = {}
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"DFA Analysis for {ticker}")
    print(f"{'='*60}")
    
    dfa_results[ticker] = dfa_analysis(X[ticker])
    
    # Plot DFA
    fig = plot_dfa(dfa_results[ticker])
    plt.savefig(f'../figures/{ticker}_dfa.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3.3 Multifractal Analysis

In [ ]:
# Multifractal analysis using absolute moments
mf_results = {}
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Multifractal Analysis for {ticker}")
    print(f"{'='*60}")
    
    mf_results[ticker] = multifractal_analysis(prices[ticker])
    
    # Plot multifractal spectrum
    fig = plot_multifractal(mf_results[ticker])
    plt.savefig(f'../figures/{ticker}_multifractal.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# MF-DFA analysis
mfdfa_results = {}
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"MF-DFA Analysis for {ticker}")
    print(f"{'='*60}")
    
    mfdfa_results[ticker] = mfdfa_analysis(X[ticker])
    
    # Plot MF-DFA
    fig = plot_mfdfa(mfdfa_results[ticker])
    plt.savefig(f'../figures/{ticker}_mfdfa.png', dpi=150, bbox_inches='tight')
    plt.show()

---
# Part 4: Granger Causality

Analyze causal relationship between the two stock returns.

In [ ]:
# Prepare bivariate data
bivariate_data = prepare_bivariate_series(X['AAPL'], X['MSFT'], 'AAPL', 'MSFT')
print(f"Bivariate data shape: {bivariate_data.shape}")

In [ ]:
# Select VARMA order
best_aic, best_bic = select_varma_order(bivariate_data, max_ar=3, max_ma=3)

In [ ]:
# Fit VARMA model
varma_result = varma_model(bivariate_data, order=best_bic)

In [ ]:
# Get coefficients and significance
coeffs = get_coefficients(varma_result)

In [ ]:
# Granger causality tests
gc_results = granger_causality_test(bivariate_data, maxlag=10, name1='AAPL', name2='MSFT')

# Plot results
fig = plot_granger_results(gc_results, 'AAPL', 'MSFT')
plt.savefig('../figures/granger_causality.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part 5: Fourier Transform and Power Spectrum

Analyze frequency domain characteristics.

In [ ]:
# Fourier transform
fft_results = {}
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Fourier Analysis for {ticker}")
    print(f"{'='*60}")
    
    fft_results[ticker] = fourier_transform(X[ticker])
    
    # Plot Fourier coefficients
    fig = plot_fourier(fft_results[ticker], title=f'{ticker} ')
    plt.savefig(f'../figures/{ticker}_fourier.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Power spectral density
psd_results = {}
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Power Spectrum for {ticker}")
    print(f"{'='*60}")
    
    psd_results[ticker] = power_spectrum(X[ticker])
    
    # Plot PSD
    fig = plot_power_spectrum(psd_results[ticker], title=f'{ticker} ')
    plt.savefig(f'../figures/{ticker}_psd.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Analyze spectral properties
    analyze_spectral_properties(psd_results[ticker], title=ticker)

In [ ]:
# Compare spectra of both stocks
fig = compare_spectra([X['AAPL'], X['MSFT']], ['AAPL', 'MSFT'], title='Stock Returns ')
plt.savefig('../figures/spectra_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part 6: Empirical Mode Decomposition

Decompose time series into Intrinsic Mode Functions.

In [ ]:
# EMD decomposition
emd_results = {}
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"EMD Analysis for {ticker}")
    print(f"{'='*60}")
    
    emd_results[ticker] = emd_decomposition(X[ticker].values)

In [ ]:
# Plot all IMFs
for ticker in TICKERS:
    fig = plot_imfs(emd_results[ticker])
    plt.savefig(f'../figures/{ticker}_emd_all.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Plot selected IMFs (c1, c_k/2, c_3k/4, c_k)
for ticker in TICKERS:
    fig = plot_selected_imfs(emd_results[ticker])
    plt.savefig(f'../figures/{ticker}_emd_selected.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Compute and plot Hurst exponents of IMFs
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"IMF Hurst Analysis for {ticker}")
    print(f"{'='*60}")
    
    analysis = analyze_imfs(emd_results[ticker])
    
    fig = plot_hurst_imf(analysis['hurst_values'])
    plt.savefig(f'../figures/{ticker}_imf_hurst.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# PSD of first two IMFs
for ticker in TICKERS:
    fig = analyze_imf_psd(emd_results[ticker]['imfs'], indices=[0, 1])
    plt.savefig(f'../figures/{ticker}_imf_psd.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Analyze reduced series (X - c1 and X - c1 - c2)
for ticker in TICKERS:
    fig = analyze_reduced_series(emd_results[ticker])
    plt.savefig(f'../figures/{ticker}_reduced_series.png', dpi=150, bbox_inches='tight')
    plt.show()

---
# Summary and Conclusions

## Key Findings

### Part 1: Data Preprocessing
- Successfully downloaded and processed daily closing prices for AAPL and MSFT
- Computed log returns and demeaned them for analysis

### Part 2: Stationarity and Autocorrelation
- ADF test results indicate stationarity of returns
- ACF/PACF patterns suggest ARMA model orders
- Nonlinear autocorrelation (ACF of |X|) shows volatility clustering

### Part 3: Fractal Behavior
- Hurst exponent values indicate persistence/anti-persistence
- DFA confirms fractal scaling behavior
- Multifractal analysis reveals complexity of the time series

### Part 4: Granger Causality
- VARMA model fitted to bivariate data
- F-tests reveal directional causality between stocks

### Part 5: Fourier Analysis
- Power spectra show frequency characteristics
- Spectral slopes indicate noise characteristics

### Part 6: EMD
- Decomposed returns into IMFs with different time scales
- Hurst exponents of IMFs show varying persistence levels
- Reduced series analysis reveals contribution of different modes